In [ ]:
# EncoderGuiSmoke.ipynb
# One-cell rotary-encoder HDMI GUI control loop with resource usage monitor.
# Only the GUI operation path remains; raw-register, live-monitor, and
# synthetic-event debug prints from earlier revisions have been removed.
#
# Note: direction / swap / debounce parameters may need adjustment for
# specific hardware. Adjust the constants in audio_lab_pynq/encoder_input.py
# or call enc.configure(...) once before the loop.

import os
import sys
import time

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "GUI")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.encoder_input import EncoderInput
from audio_lab_pynq.encoder_ui import EncoderUiController
from audio_lab_pynq.hdmi_backend import (
    AudioLabHdmiBackend,
    DEFAULT_WIDTH,
    DEFAULT_HEIGHT,
)
from audio_lab_pynq.hdmi_state.resource_sampler import ResourceSampler
from GUI.compact_v2.state import AppState
from GUI.pynq_multi_fx_gui import render_frame_800x480_compact_v2


def _find_encoder_ip_name(overlay):
    ip_dict = getattr(overlay, "ip_dict", {})
    for name in ("enc_in_0/s_axi", "enc_in_0",
                 "axi_encoder_input_0", "axi_encoder_input"):
        if name in ip_dict:
            return name
    for name in ip_dict:
        lower = name.lower()
        if "encoder" in lower or lower.startswith("enc_in"):
            return name
    raise RuntimeError("encoder IP not found in overlay.ip_dict")


# --- Overlay attach (once per kernel) -------------------------------------
if "ovl" not in globals():
    ovl = AudioLabOverlay()

# --- Encoder + GUI wiring -------------------------------------------------
enc = EncoderInput.from_overlay(ovl, ip_name=_find_encoder_ip_name(ovl))
enc.configure(clear_on_read=True)

state = AppState()
controller = EncoderUiController(state, overlay=ovl, apply_on_value_change=False)
backend = AudioLabHdmiBackend(ovl, width=DEFAULT_WIDTH, height=DEFAULT_HEIGHT)
backend.start()

# --- Run-loop settings ----------------------------------------------------
GUI_SECONDS = 120
GUI_FPS = 5
STATUS_INTERVAL_S = 2.0
RUN_ENCODER_GUI = True


def stop_encoder_gui():
    """Call from a later cell to break out of the loop cleanly."""
    global RUN_ENCODER_GUI
    RUN_ENCODER_GUI = False


def _fmt_pct(value):
    return ("%5.1f%%" % value) if value is not None else "  n/a"


def _fmt_temp(value):
    return ("%4.1fC" % value) if value is not None else "  n/a"


sampler = ResourceSampler()
sampler.sample()  # bootstrap (first call returns None for CPU%)

period = 1.0 / max(1.0, float(GUI_FPS))
t0 = time.time()
last_status_t = 0.0
last_status_frames = 0
frames = 0

try:
    while RUN_ENCODER_GUI and (time.time() - t0) < GUI_SECONDS:
        loop_t = time.time()
        events = enc.poll(timestamp=loop_t - t0)
        if events:
            controller.handle_events(events)
        state.t = loop_t - t0
        frame = render_frame_800x480_compact_v2(state)
        backend.write_frame(frame, placement="manual", offset_x=0, offset_y=0)
        frames += 1
        if (loop_t - last_status_t) >= STATUS_INTERVAL_S:
            r = sampler.sample()
            dt = (loop_t - last_status_t) if last_status_t > 0 else max(1e-3, loop_t - t0)
            fps = (frames - last_status_frames) / dt
            rss_mb = int(r.get("proc_rss_kb", 0)) // 1024
            mem_total_mb = int(r.get("mem_total_kb", 0)) // 1024
            mem_avail_mb = int(r.get("mem_avail_kb", 0)) // 1024
            mem_used_mb = mem_total_mb - mem_avail_mb if mem_total_mb else 0
            mem_used_pct = (100.0 * mem_used_mb / mem_total_mb) if mem_total_mb else None
            print(
                "t=%6.1fs fps=%4.1f sys_cpu=%s proc_cpu=%s "
                "mem=%4d/%4dMB(%s) rss=%4dMB temp=%s"
                % (
                    loop_t - t0, fps,
                    _fmt_pct(r.get("sys_cpu_pct")),
                    _fmt_pct(r.get("proc_cpu_pct")),
                    mem_used_mb, mem_total_mb, _fmt_pct(mem_used_pct),
                    rss_mb,
                    _fmt_temp(r.get("temperature_c")),
                )
            )
            last_status_t = loop_t
            last_status_frames = frames
        elapsed = time.time() - loop_t
        if elapsed < period:
            time.sleep(period - elapsed)
except KeyboardInterrupt:
    pass
finally:
    backend.stop()
